# Notebook 5: Paper Figures
**Paper 5 – Cross-Framework Quantum Algorithm Benchmarking**

Generates all 15 paper figures:
- Figs. 2–4: Structural metrics (gate count, depth, composition)
- Fig. 5: GHZ scaling
- Figs. 6–8: Performance (compile time, sim time, memory)
- Figs. 9–10: Statistical tests (JSD heatmap, chi-squared)
- Figs. 11–12: Quantum Volume
- Fig. 13: Bell State histograms
- Fig. 14: Grover's scaling
- **Fig. 15**: QPack radar chart (NEW — from QPack §IV-F)

**Prerequisites**: Run notebooks 1–4 first.

**Outputs**: `benchmarks/results/structural/*.pdf/*.png`

In [ ]:
import os, sys
QCANVAS_ROOT = os.path.abspath('../..')
if QCANVAS_ROOT not in sys.path:
    sys.path.insert(0, QCANVAS_ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from benchmarks.scripts.figure_styles import (
    apply_paper_style, save_figure,
    plot_grouped_bar, plot_scaling_lines,
    plot_jsd_heatmap, plot_measurement_histogram,
    plot_radar, framework_legend_patches,
    FRAMEWORK_COLORS, FRAMEWORK_LABELS, FRAMEWORK_MARKERS
)

apply_paper_style()
print('Figure styles loaded. Generating all 15 paper figures...')

In [ ]:
# Load all data
df_struct = pd.read_csv('../metrics/structural_metrics.csv')
df_ctimes = pd.read_csv('../metrics/compilation_times.csv')
df_sim    = pd.read_csv('../metrics/simulation_metrics.csv')
df_qv     = pd.read_csv('../metrics/quantum_volume_estimates.csv')
df_tests  = pd.read_csv('../metrics/statistical_tests.csv')
df_qpack  = pd.read_csv('../metrics/qpack_scores.csv')

# Baseline: minimum qubit count per algorithm
df_fixed  = df_struct[df_struct['n_qubits'] == df_struct.groupby('algorithm')['n_qubits'].transform('min')]
df_4k     = df_sim[(df_sim['shots'] == 4096) & df_sim['success']]
df_4k_fix = df_4k[df_4k['n_qubits'] == df_4k.groupby('algorithm')['n_qubits'].transform('min')]

print('Data loaded successfully.')

In [ ]:
# ── Fig. 2: Gate count ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
plot_grouped_bar(ax, df_fixed, 'algorithm', 'total_gates',
                 title='Fig. 2 — Total Gate Count per Algorithm and Framework',
                 ylabel='Total Gates')
plt.tight_layout(); save_figure(fig, 'fig02_gate_counts', 'structural'); plt.show()

In [ ]:
# ── Fig. 3: Circuit depth ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
plot_grouped_bar(ax, df_fixed, 'algorithm', 'circuit_depth',
                 title='Fig. 3 — Circuit Depth per Algorithm and Framework',
                 ylabel='Circuit Depth')
plt.tight_layout(); save_figure(fig, 'fig03_circuit_depth', 'structural'); plt.show()

In [ ]:
# ── Fig. 5: GHZ scaling ──────────────────────────────────────────────────────
df_ghz = df_struct[df_struct['algorithm'] == 'ghz_state']
fig, ax = plt.subplots(figsize=(8, 5))
plot_scaling_lines(ax, df_ghz, title='Fig. 5 — GHZ State Gate Count Scaling (3–8 qubits)',
                   ylabel='Total Gate Count')
plt.tight_layout(); save_figure(fig, 'fig05_ghz_scaling', 'structural'); plt.show()

In [ ]:
# ── Fig. 9: JSD Heatmap (algorithm-level aggregate) ──────────────────────────
jsd_cols = ['jsd_qiskit_cirq', 'jsd_qiskit_pl', 'jsd_cirq_pl']
df_heat = df_tests.groupby('algorithm', as_index=True)[jsd_cols].mean()
df_heat = df_heat[['jsd_qiskit_cirq', 'jsd_qiskit_pl', 'jsd_cirq_pl']]
df_heat.columns = ['Qiskit↔Cirq', 'Qiskit↔PennyLane', 'Cirq↔PennyLane']

fig, ax = plt.subplots(figsize=(8, max(6, len(df_heat) * 0.45 + 1.2)))
plot_jsd_heatmap(ax, df_heat, title='Fig. 9 — Pairwise JSD Heatmap (Algorithm-Level, 4096 shots)')
plt.tight_layout(); save_figure(fig, 'fig09_jsd_heatmap', 'simulation'); plt.show()
print(f'Fig. 9 rows: {len(df_heat)} algorithms (aggregated from {len(df_tests)} records).')

In [ ]:
# ── Fig. 10: Distribution equivalence summary (algorithm-level) ───────────────
import seaborn as sns

jsd_cols = ['jsd_qiskit_cirq', 'jsd_qiskit_pl', 'jsd_cirq_pl']
df_jsd_alg = df_tests.groupby('algorithm', as_index=True)[jsd_cols].mean()

def to_label(v):
    if v <= 0.01:
        return '✓'
    if v <= 0.05:
        return '~'
    return '✗'

df_labels = df_jsd_alg.applymap(to_label)
label_map = {'✓': 1.0, '~': 0.5, '✗': 0.0}
df_sig = df_labels.applymap(lambda x: label_map[x])
df_sig.columns = ['Qiskit↔Cirq', 'Qiskit↔PL', 'Cirq↔PL']
annot_vals = df_labels.values

fig, ax = plt.subplots(figsize=(7, max(5, len(df_sig) * 0.45 + 1)))
sns.heatmap(
    df_sig, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
    annot=annot_vals, fmt='', linewidths=0.5
    )
ax.set_title('Fig. 10 — Distribution Equivalence Summary (Algorithm-Level: ✓/~/✗)')
plt.tight_layout()
save_figure(fig, 'fig10_equivalence_summary', 'simulation')
plt.show()
print(f'Fig. 10 rows: {len(df_sig)} algorithms (aggregated from {len(df_tests)} records).')

In [ ]:
# ── Fig. 14: Grover's scaling ────────────────────────────────────────────────
df_grover = df_struct[df_struct['algorithm'] == 'grovers_algorithm']
fig, ax = plt.subplots(figsize=(8, 5))
plot_scaling_lines(ax, df_grover, title="Fig. 14 — Grover's Gate Count Scaling (2–6 qubits)",
                   ylabel='Total Gate Count')
plt.tight_layout(); save_figure(fig, 'fig14_grovers_scaling', 'structural'); plt.show()

In [ ]:
# ── Fig. 15: QPack Radar Chart (NEW) ─────────────────────────────────────────
if not df_qpack.empty:
    scores_dict = {}
    for _, row in df_qpack.iterrows():
        scores_dict[row['framework']] = {
            'S_compile_speed': row.get('S_compile_speed', 0),
            'S_scalability':   row.get('S_scalability', 0),
            'S_accuracy':      row.get('S_accuracy', 0),
            'S_capacity':      row.get('S_capacity', 0),
        }

    fig = plt.figure(figsize=(8, 8))
    ax  = fig.add_subplot(111, projection='polar')
    plot_radar(ax, scores_dict)
    plt.tight_layout()
    save_figure(fig, 'fig15_qpack_radar', 'simulation')
    plt.show()
    print('\nFig. 15 (QPack Radar) saved.')
else:
    print('[SKIP] qpack_scores.csv not found — run nb03_statistical_analysis.ipynb first')